In [19]:
ETF = ["SPY", "QQQ", "DIA"]
Stock = ["AAPL", "ACMR", "AMD", "GOOG", "INTC", "META", "TSLA"]

In [20]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make notebook runnable from either repo root or Stat_result directory.
if (Path.cwd() / "DataSet").exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Agent_hardThreshold_Class import Agent_hardThreshold
from Agent_Percentile_Class import Agent_Percentile
from Agent_Garch_Class import Agent_Garch
from Agent_LongTerm_Class import Agent_LongTerm

In [21]:
def build_agents():
    """Return a fresh set of 4 agents with canonical parameters."""
    return [
        Agent_hardThreshold(display_name="Agent_hardThreshold", k=1,
                            allow_short=True, delta_hedge=False),
        Agent_Percentile(display_name="Agent_Perc", entry_percentile=0.2,
                         allow_short=True, delta_hedge=False),
        Agent_Garch(display_name="Agent_Garch", entry_threshold=1,
                    allow_short=True, delta_hedge=False),
        Agent_LongTerm(display_name="Agent_LongTerm", entry_threshold=1,
                       long_term_window=60, allow_short=True, delta_hedge=False),
    ]


def load_data(ticker: str) -> pd.DataFrame:
    path = PROJECT_ROOT / "DataSet" / f"{ticker}.csv"
    data = pd.read_csv(path)
    data["VRP_20d_mean"] = data["VRP"].rolling(window=20, min_periods=20).mean()
    data["VRP_20d_std"]  = data["VRP"].rolling(window=20, min_periods=20).std()
    return data


def compute_stats(agents, data) -> pd.DataFrame:
    rows = []
    for agent in agents:
        res = agent.get_result()
        r = pd.Series(res["return"], dtype=float)
        r = r.reindex(range(len(data))).fillna(0.0)
        r = r.replace([np.inf, -np.inf], np.nan).fillna(0.0)

        n            = len(r)
        mean_r       = r.mean()
        std_r        = r.std(ddof=1) if n > 1 else 0.0
        downside     = r[r < 0]
        downside_std = downside.std(ddof=1) if len(downside) > 1 else 0.0

        # Transaction-based win rate (same logic as Visual.STATS)
        pos_state = pd.Series(res.get("position_state_for_pnl", []), dtype=float)
        if len(pos_state) == len(r) and len(pos_state) > 0:
            state_changes = pos_state.diff().ne(0).cumsum()
            trade_returns = [
                g.sum() for _, g in r.groupby(state_changes)
                if pos_state.loc[g.index[0]] != 0
            ]
            win_rate = (sum(1 for x in trade_returns if x > 0) / len(trade_returns)
                        if trade_returns else 0.0)
        else:
            trading_days = r[r != 0]
            win_rate = (trading_days > 0).mean() if len(trading_days) > 0 else 0.0

        equity        = np.exp(r.cumsum())
        annual_return = np.exp(mean_r * 252.0) - 1.0
        annual_vol    = std_r * np.sqrt(252.0)
        sharpe        = (mean_r / std_r)        * np.sqrt(252.0) if std_r        > 0 else np.nan
        sortino       = (mean_r / downside_std)  * np.sqrt(252.0) if downside_std > 0 else np.nan
        max_dd        = (equity / equity.cummax() - 1.0).min()

        rows.append({
            "Agent":         res["display_name"],
            "Win Rate":      round(win_rate,      4),
            "Sharpe":        round(sharpe,        4) if pd.notna(sharpe)  else np.nan,
            "Sortino":       round(sortino,       4) if pd.notna(sortino) else np.nan,
            "Ann. Return":   round(annual_return, 4),
            "Ann. Vol":      round(annual_vol,    4),
            "Max Drawdown":  round(max_dd,        4),
        })
    return pd.DataFrame(rows)


def compute_greeks_attr_pct(agents) -> pd.DataFrame:
    """Return signed Greek attribution percentages by agent (denominator = sum(abs(bucket)))."""
    greek_names = ["delta", "gamma", "vega", "theta", "vanna", "volga", "rho", "residual"]
    rows = []

    for agent in agents:
        res = agent.get_result()
        attr = res.get("greeks_attribute", {})

        totals = {
            g: float(np.nansum(pd.Series(attr.get(g, []), dtype=float)))
            for g in greek_names
        }
        total_abs = float(np.sum(np.abs(list(totals.values()))))

        row = {"Agent": res["display_name"]}
        for g in greek_names:
            pct = (totals[g] / total_abs * 100.0) if total_abs > 1e-12 else 0.0
            row[f"{g.capitalize()} %"] = round(pct, 2)
        rows.append(row)

    return pd.DataFrame(rows)


def run_backtest(tickers: list) -> tuple[pd.DataFrame, pd.DataFrame]:
    all_stats = []
    all_greek_pct = []

    for ticker in tickers:
        print(f"  {ticker}...", end=" ", flush=True)
        data   = load_data(ticker)
        agents = build_agents()
        for _, row in data.iterrows():
            for agent in agents:
                agent.trade(row)

        stats = compute_stats(agents, data)
        stats.insert(0, "Ticker", ticker)
        all_stats.append(stats)

        greek_pct = compute_greeks_attr_pct(agents)
        greek_pct.insert(0, "Ticker", ticker)
        all_greek_pct.append(greek_pct)

        print("done")

    return (
        pd.concat(all_stats, ignore_index=True),
        pd.concat(all_greek_pct, ignore_index=True),
    )

In [22]:
print("=== ETF backtest ===")
etf_stats, etf_greek_pct = run_backtest(ETF)
etf_stats.to_csv("etf_stats.csv", index=False)
etf_greek_pct.to_csv("etf_greek_attr_pct.csv", index=False)

print("\n[ETF] Performance table")
print(etf_stats.to_string(index=False))

print("\n[ETF] Greeks attribution (%) table")
etf_greek_pct_display = etf_greek_pct.copy()
for c in etf_greek_pct_display.columns:
    if c.endswith("%"):
        etf_greek_pct_display[c] = etf_greek_pct_display[c].map(lambda x: f"{x:+.2f}%")
print(etf_greek_pct_display.to_string(index=False))

print("\nSaved -> etf_stats.csv")
print("Saved -> etf_greek_attr_pct.csv")

=== ETF backtest ===
  SPY... done
  QQQ... done
  DIA... done

[ETF] Performance table
Ticker               Agent  Win Rate  Sharpe  Sortino  Ann. Return  Ann. Vol  Max Drawdown
   SPY Agent_hardThreshold    0.7000  1.2646   1.3248       5.7974    1.5155       -0.7231
   SPY          Agent_Perc    0.6923  1.0237   1.1120       3.9001    1.5524       -0.7231
   SPY         Agent_Garch    0.6809  1.6241   1.6751      11.2763    1.5440       -0.7002
   SPY      Agent_LongTerm    0.6957  0.5235   0.4641       1.3097    1.5991       -0.9109
   QQQ Agent_hardThreshold    0.7297  0.8025   0.6644       2.5360    1.5739       -0.8346
   QQQ          Agent_Perc    0.6410  0.2067   0.1965       0.4273    1.7213       -0.8675
   QQQ         Agent_Garch    0.6316  0.4221   0.3595       0.9469    1.5785       -0.8209
   QQQ      Agent_LongTerm    0.7333  1.6349   1.5359      10.3727    1.4870       -0.6117
   DIA Agent_hardThreshold    0.7500  3.2397   3.8718     119.7859    1.4798       -0.5029
  

In [23]:
print("=== Stock backtest ===")
stock_stats, stock_greek_pct = run_backtest(Stock)
stock_stats.to_csv("stock_stats.csv", index=False)
stock_greek_pct.to_csv("stock_greek_attr_pct.csv", index=False)

print("\n[Stock] Performance table")
print(stock_stats.to_string(index=False))

print("\n[Stock] Greeks attribution (%) table")
stock_greek_pct_display = stock_greek_pct.copy()
for c in stock_greek_pct_display.columns:
    if c.endswith("%"):
        stock_greek_pct_display[c] = stock_greek_pct_display[c].map(lambda x: f"{x:+.2f}%")
print(stock_greek_pct_display.to_string(index=False))

print("\nSaved -> stock_stats.csv")
print("Saved -> stock_greek_attr_pct.csv")

=== Stock backtest ===
  AAPL... done
  ACMR... done
  AMD... done
  GOOG... done
  INTC... done
  META... done
  TSLA... done

[Stock] Performance table
Ticker               Agent  Win Rate  Sharpe  Sortino  Ann. Return  Ann. Vol  Max Drawdown
  AAPL Agent_hardThreshold    0.6585  0.5886   0.5079       1.4497    1.5221       -0.7045
  AAPL          Agent_Perc    0.6111 -0.0772  -0.0629      -0.1091    1.4974       -0.8679
  AAPL         Agent_Garch    0.6222  0.4594   0.3848       1.0009    1.5097       -0.6940
  AAPL      Agent_LongTerm    0.5778  0.4970   0.3988       0.9776    1.3721       -0.6382
  ACMR Agent_hardThreshold    0.5556  0.2839   0.1919       0.4981    1.4237       -0.6453
  ACMR          Agent_Perc    0.6923  1.3969   1.2759      13.6202    1.9202       -0.6453
  ACMR         Agent_Garch    0.7000  1.2594   1.5129       3.8847    1.2595       -0.5268
  ACMR      Agent_LongTerm    0.2000 -0.9362  -0.4292      -0.7287    1.3936       -0.6576
   AMD Agent_hardThreshold 